# Benchmark k-NN po redukcji wymiarowości do 3D (PCA)

Ten notatnik powtarza **dokładnie ten sam** benchmark k-NN co w sprincie 1 (sanity plots +
skan czasu wykonania dla wszystkich metod i rozmiarów danych), z jedną różnicą: przed
uruchomieniem każdej metody cechy są najpierw rzutowane z oryginalnej przestrzeni (5D) do
**3 wymiarów za pomocą PCA** (liniowa redukcja wymiarowości, w przeciwieństwie do
nieliniowego UMAP i w przeciwieństwie do zero-paddingu ze sprintu 2 -- tu wymiarów faktycznie
ubywa, a nie przybywa, więc część informacji zostaje utracona).

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import numpy as np

notebook_dir = Path().resolve()
project_root = notebook_dir.parent
src_path = str(project_root / "src")
if src_path not in sys.path:
    sys.path.append(src_path)

print(f"Katalog roboczy (notatnik): {notebook_dir}")
print(f"Katalog główny projektu: {project_root}")
print(f"Załadowano ścieżkę modułów: {src_path}")

In [ ]:
from hep_tracking.models import (
    knn_numpy_brute_force,
    knn_cupy_brute_force,
    knn_scipy_ckdtree,
    knn_sklearn_kdtree,
    knn_sklearn_balltree,
)
from hep_tracking.plots import plot_3d_hits, plot_distance_distributions
from hep_tracking.utils import measure_execution_time

print("Zależności modułu hep_tracking załadowane pomyślnie!")
from sklearn.decomposition import PCA


def reduce_to_3d_pca(X, n_components=3, random_state=42):
    """Redukuje macierz cech X do n_components wymiarów za pomocą PCA.

    W odróżnieniu od UMAP, PCA jest rzutowaniem liniowym i działa praktycznie
    natychmiastowo nawet dla N=1M, więc nie ma potrzeby cache'owania wyniku na dysku.
    Zwraca zredukowane cechy oraz udział wyjaśnionej wariancji (informacja o tym,
    ile ze zmienności oryginalnych danych zachowały 3 pierwsze składowe główne).
    """
    pca = PCA(n_components=n_components, random_state=random_state)
    X_reduced = pca.fit_transform(X)
    explained = pca.explained_variance_ratio_.sum()
    return X_reduced, explained


## Wykresy diagnostyczne (sanity plots) na danych po PCA

Zanim uruchomimy pełny benchmark, sprawdzamy wizualnie, jak wygląda zbiór kontrolny
(`dataset_hard_10k`) po zrzutowaniu do 3D przez PCA -- czy struktura torów jest nadal
widoczna, oraz ile wariancji oryginalnych danych zachowały 3 główne składowe.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

data_dir = project_root / "data"
sanity_dataset = data_dir / "dataset_hard_10k.npz"

if sanity_dataset.exists():
    loaded_data = np.load(sanity_dataset)
    features_sanity_full = loaded_data["X"]
    labels_sanity = loaded_data["y"]

    print(f"Wczytano zbiór kontrolny: {sanity_dataset.name}")
    print(f"Kształt oryginalnej macierzy cech: {features_sanity_full.shape}")

    features_sanity_pca, explained_sanity = reduce_to_3d_pca(features_sanity_full)
    print(f"Kształt macierzy cech po PCA: {features_sanity_pca.shape}")
    print(f"Wyjaśniona wariancja (3 składowe): {explained_sanity:.2%}")

    plot_3d_hits(features_sanity_pca, labels_sanity)
    plt.show()

    plot_distance_distributions(features_sanity_pca, labels_sanity)
    plt.show()
else:
    print(f"Nie znaleziono pliku {sanity_dataset.name}. Upewnij się, że wygenerowałeś dane (make generate).")

**Wnioski z wykresów diagnostycznych (PCA 3D):**

- Jeśli wyjaśniona wariancja jest bliska 100%, oznacza to, że oryginalne 5 wymiarów było w
  dużej mierze redundantne (np. silnie skorelowane ze sobą) i redukcja do 3D nie traci
  prawie żadnej informacji -- tory powinny wyglądać tak samo wyraźnie jak w oryginalnej
  przestrzeni.
- Jeśli wyjaśniona wariancja jest wyraźnie poniżej 100%, PCA odrzuciło część rzeczywistej
  zmienności danych (np. informację o czasie/warstwie, jeśli nie była silnie skorelowana z
  pozycją) -- wtedy należy się spodziewać, że histogram odległości Same Track / Cross Track
  będzie miał **większe nakładanie się** niż w oryginalnej przestrzeni, bo część sygnału
  odróżniającego pary zniknęła wraz z odrzuconymi składowymi.
- W przeciwieństwie do zero-paddingu (sprint 2, 5D→8D, bezstratny) PCA 5D→3D jest z
  definicji **stratne** (tracimy wymiary, a nie dodajemy neutralnych zer) -- to fundamentalna
  różnica, o której trzeba pamiętać przy interpretacji wyników poniżej.

## Benchmark k-NN na danych zredukowanych do 3D (PCA)

Konfiguracja identyczna jak w oryginalnym benchmarku sprintu 1: te same rozmiary danych
$N \in \{1\text{k}, 10\text{k}, 100\text{k}, 1\text{M}\}$, oba tryby (Easy/Hard), $k=5$ i te
same pięć implementacji k-NN. Jedyna zmiana: każdy załadowany zbiór jest najpierw redukowany
do 3D przez PCA, zanim trafi do którejkolwiek z metod.

In [ ]:
target_sizes = {"1k": 1000, "10k": 10000, "100k": 100000, "1M": 1000000}
dataset_modes = ["easy", "hard"]
k_neighbors = 5

methods = {
    "Numpy Brute Force": knn_numpy_brute_force,
    "CuPy Brute Force": knn_cupy_brute_force,
    "Scipy cKDTree": knn_scipy_ckdtree,
    "Sklearn KDTree": knn_sklearn_kdtree,
    "Sklearn BallTree": knn_sklearn_balltree,
}

results = {mode: {method: [] for method in methods} for mode in dataset_modes}

In [ ]:
import warnings

warnings.filterwarnings('ignore')

explained_variance_log = {}

for mode in dataset_modes:
    print(f"\n{'='*45}")
    print(f" TRYB DANYCH: {mode.upper()}")
    print(f"{'='*45}")

    for size_label, size_val in target_sizes.items():
        filename = data_dir / f"dataset_{mode}_{size_label}.npz"

        if not filename.exists():
            print(f"\n[BRAK PLIKU] {filename.name} - Pomijam.")
            for method_name in methods:
                results[mode][method_name].append(None)
            continue

        loaded_data = np.load(filename)
        features_full = loaded_data["X"]
        print(f"\n Załadowano: {size_label} ({features_full.shape[0]} hitów, {features_full.shape[1]}D oryginalnie)")

        features, explained = reduce_to_3d_pca(features_full)
        explained_variance_log[(mode, size_label)] = explained
        print(f"  > PCA: {features_full.shape[1]}D -> {features.shape[1]}D (wyjaśniona wariancja: {explained:.2%})")

        for method_name, method_callable in methods.items():

            if method_name in ["Numpy Brute Force", "CuPy Brute Force", "Sklearn BallTree"] and size_val > 100000:
                print(f"  > Pomijam: {method_name} (Zbyt wolne dla {size_label})")
                results[mode][method_name].append(None)
                continue

            def benchmark_wrapper():
                method_callable(features, k_neighbors)

            min_time = measure_execution_time(benchmark_wrapper, num_runs=3, warmup_runs=1)
            results[mode][method_name].append(min_time)

            print(f"  > {method_name}: {min_time * 1000:.2f} ms")

print("\nBenchmark (PCA 3D) zakończony sukcesem!")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
numeric_sizes = list(target_sizes.values())

for idx, mode in enumerate(dataset_modes):
    ax = axes[idx]
    
    for method_name, method_times in results[mode].items():
        valid_sizes = [s for s, t in zip(numeric_sizes, method_times) if t is not None]
        valid_times = [t for t in method_times if t is not None]
        
        if valid_times:
            ax.plot(
                valid_sizes,
                valid_times,
                marker="o",
                label=method_name,
                linewidth=2,
                markersize=6,
            )

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_title(f"Skalowalność kNN (PCA 3D) - tryb: {mode.upper()}", fontsize=14)
    ax.set_xlabel("Liczba punktów (N)", fontsize=12)
    
    if idx == 0:
        ax.set_ylabel("Minimalny czas wykonania (s)", fontsize=12)
        
    ax.grid(True, which="both", ls="--", alpha=0.5)
    ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

plt.suptitle("Benchmark kNN po redukcji wymiarowości do 3D (PCA)", fontsize=15, y=1.03)

## Wnioski

- **Wpływ na czas wykonania:** przy mniejszej liczbie wymiarów (3D zamiast 5D) każda metoda
  liczy mniej operacji na parę punktów (mniej różnic/mnożeń przy obliczaniu odległości), więc
  spodziewamy się, że wszystkie metody będą co najmniej nieznacznie szybsze niż w oryginalnym
  benchmarku z 5D -- najbardziej powinny to odczuć metody brute-force (koszt liczenia
  odległości skaluje się liniowo z liczbą wymiarów), a najmniej `cKDTree`, który już w 5D był
  bardzo szybki i zdominowany raczej przez $\log N$ niż przez wymiarowość.
- **Ranking metod powinien pozostać ten sam co w 5D** -- redukcja wymiarowości nie zmienia
  charakteru skalowania względem $N$ (brute-force nadal $O(N)$, drzewa nadal $\sim O(\log N)$
  w niskiej wymiarowości) -- zmienia się tylko stała multiplikatywna, nie kształt krzywej.
- **Uwaga na koszt informacyjny:** w przeciwieństwie do eksperymentu z UMAP czy do
  zero-paddingu w sprincie 2, PCA do 3D jest \emph{stratne z definicji}, jeśli wyjaśniona
  wariancja (wypisywana powyżej dla każdego zbioru) jest wyraźnie poniżej 100%. Szybszy
  benchmark k-NN nic nie znaczy, jeśli sąsiedzi znalezieni w zredukowanej przestrzeni
  przestają odpowiadać prawdziwym sąsiadom sprzed redukcji -- to należałoby dodatkowo
  zweryfikować przez recall względem oryginalnego (5D) ground truth, analogicznie do tego,
  jak to zrobiono dla metod ANN w sprincie 2.
- **Praktyczny wniosek:** ten eksperyment mówi nam głównie o **koszcie obliczeniowym**
  redukcji wymiarowości na k-NN, a nie o jej **jakości** -- do pełnej oceny, czy PCA 3D
  nadaje się do dalszych etapów pipeline'u (np. klasyfikacji par), potrzebny byłby dodatkowy
  pomiar recall analogiczny do tego z sanity plots wyżej.